In [ ]:
# Lab type: write
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: Building Preprocessing Pipelines
# Task: Assemble a sklearn Pipeline from scratch, evaluate it honestly with
#       cross-validation, inspect fitted transformer state, and serialise it for deployment.

# Lab: Building a Preprocessing Pipeline

This lab mirrors Lesson 2 end-to-end. You will build a `Pipeline` from scratch —
not scaffold it with AI, write it yourself — so you know exactly what each step
does and can catch the silent bugs that generated code introduces.

Work through each step in order. Stub cells have `pass` or `# your code here`;
replace them with working code.

**Outputs are cleared.** Run each cell to see the result.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install scikit-learn pandas numpy joblib --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import joblib

np.random.seed(42)
n = 800

tenure_days     = np.random.exponential(300, n).clip(30, 1000).astype(float)
monthly_spend   = np.random.lognormal(4.5, 0.6, n)
support_tickets = np.random.poisson(2, n).clip(0, 15).astype(float)
plan_encoded    = np.random.choice([0.0, 1.0, 2.0], n, p=[0.4, 0.4, 0.2])

# Introduce ~8% missing values to make imputation relevant
for col_arr in [tenure_days, monthly_spend, support_tickets]:
    mask = np.random.rand(n) < 0.08
    col_arr[mask] = np.nan

logit = np.where(
    np.isnan(tenure_days), 0, -0.003 * tenure_days
) + np.where(
    np.isnan(support_tickets), 0, 0.4 * support_tickets
) + np.where(
    np.isnan(monthly_spend), 0, -0.02 * monthly_spend
) + 0.5 * plan_encoded
churn_prob = 1 / (1 + np.exp(-logit))
churned = (np.random.rand(n) < churn_prob).astype(int)

df = pd.DataFrame({
    "monthly_spend_log": np.log1p(np.where(np.isnan(monthly_spend), np.nan, monthly_spend)),
    "support_tickets":   support_tickets,
    "tenure_days":       tenure_days,
    "plan_encoded":      plan_encoded,
    "churned":           churned,
})

print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Churn rate: {df['churned'].mean():.1%}")

## Step 1: Split the data

Define `X` and `y`, then create `X_train`, `X_test`, `y_train`, `y_test` using
a 80/20 split with `random_state=42`.

The split must happen **before** any transformer is fitted.

In [ ]:
feature_cols = ["monthly_spend_log", "support_tickets", "tenure_days", "plan_encoded"]

X = df[feature_cols]
y = df["churned"]

# your code here: split into X_train, X_test, y_train, y_test
# X_train, X_test, y_train, y_test = ...

pass

## Step 2: Build the Pipeline

Create a `Pipeline` named `pipeline` with these three named steps:

| Name | Transformer / Estimator |
|---|---|
| `"imputer"` | `SimpleImputer(strategy="median")` |
| `"scaler"` | `StandardScaler()` |
| `"model"` | `LogisticRegression(max_iter=1000)` |

The names matter — you'll use them in Step 4.

In [ ]:
# your code here: build the pipeline
# pipeline = Pipeline([...])

pass

## Step 3: Fit and evaluate

Fit the pipeline on `X_train` and `y_train` in one call.
Then compute the AUC on `X_test`.

In [ ]:
# your code here: fit the pipeline
# pipeline.fit(...)

# your code here: predict probabilities and compute AUC
# y_proba = ...
# auc = ...
# print(f"Test AUC: {auc:.3f}")

pass

## Step 4: Cross-validate correctly

Use `cross_val_score` to estimate performance across 5 folds.
Pass the **pipeline** — not pre-scaled data — to `cross_val_score`.

Print the per-fold scores and the mean ± std.

In [ ]:
# your code here: cross-validate the pipeline
# cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc")
# print(cv_scores.round(3))
# print(f"Mean AUC: {cv_scores.mean():.3f} \u00b1 {cv_scores.std():.3f}")

pass

> **Question:** If you had scaled `X` before calling `cross_val_score` and passed
> the pre-scaled array instead of the pipeline, which folds would be contaminated
> and how?
>
> *(Write your answer here.)*

## Step 5: Inspect fitted transformer state

After fitting, access the trained imputer and scaler through `pipeline.named_steps`
and print the statistics they learned from the training data.

In [ ]:
# your code here: print the median values the imputer learned
# print("Imputer medians:", pipeline.named_steps["imputer"].statistics_)

# your code here: print the mean and standard deviation the scaler learned
# print("Scaler means:  ", pipeline.named_steps["scaler"].mean_)
# print("Scaler scales: ", pipeline.named_steps["scaler"].scale_)

pass

> **Question:** These statistics come from the training data only. Why is it important
> that the scaler uses training-set means and standard deviations — not test-set values —
> when it transforms `X_test`?
>
> *(Write your answer here.)*

## Step 6: Serialise and load

Serialise the fitted pipeline to `churn_pipeline.pkl` with `joblib.dump`.
Load it back into a new variable and verify it produces the same predictions.

In [ ]:
# your code here: save the fitted pipeline
# joblib.dump(pipeline, "churn_pipeline.pkl")

# your code here: load and compare predictions
# loaded = joblib.load("churn_pipeline.pkl")
# auc_loaded = roc_auc_score(y_test, loaded.predict_proba(X_test)[:, 1])
# print(f"AUC from loaded pipeline: {auc_loaded:.3f}")

pass

> **Question:** The loaded pipeline transforms `X_test` using the fitted scaler
> and imputer from training. You did not re-fit anything. Why is this the correct
> behaviour for a deployed model?
>
> *(Write your answer here.)*

## Challenge: catch the silent bug

The cell below uses `make_pipeline` but passes pre-imputed data to `cross_val_score`.
Identify the leakage, explain it, and rewrite the cell correctly.

In [ ]:
from sklearn.pipeline import make_pipeline

# Review this code — is it correct?
imputer_pre = SimpleImputer(strategy="median")
X_imputed = imputer_pre.fit_transform(X)   # ← imputed before CV split

pipeline_challenge = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
cv_challenge = cross_val_score(pipeline_challenge, X_imputed, y, cv=5, scoring="roc_auc")
print(f"CV AUC (challenge): {cv_challenge.mean():.3f} \u00b1 {cv_challenge.std():.3f}")

**Diagnosis:** Which step is outside the pipeline, and why does it contaminate every validation fold?

*(Write your answer here.)*

In [ ]:
# Fix: move the imputer inside the pipeline so each fold re-fits it on training rows only

# your code here
pass

## Summary

> **Answer each question in one sentence.**

1. What does `Pipeline.fit(X_train, y_train)` do to each transformer in the chain?
2. What does `Pipeline.predict(X_test)` call on each transformer, and why is that significant?
3. Why must the pipeline object (not pre-processed data) be passed to `cross_val_score`?
4. What is preserved in `churn_pipeline.pkl` that makes the loaded pipeline production-ready?